In [1]:
import torch
import torch.nn as nn

class PatchEmbedding(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=768):
        super().__init__()
        self.patch_size = patch_size
        self.conv = nn.Conv2d(in_channels = in_channels, out_channels = embed_dim, kernel_size = patch_size, stride = patch_size)

    def forward(self, x):
        x = self.conv(x).flatten(2).transpose(1,2)
        return x

In [2]:
class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, seq_len):
        super().__init__()
        self.pos_embed = nn.Parameter(torch.randn(1, seq_len + 1, embed_dim))

    def forward(self, x):
        return x + self.pos_embed

In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim = embed_dim, num_heads = num_heads, batch_first = True)

    def forward(self, x):
        return self.attn(x, x, x)[0]

In [4]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_dim):
        super().__init__()
        self.attn = MultiHeadAttention(embed_dim, num_heads)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, embed_dim)
        )
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

In [5]:
class VisionTransformer(nn.Module):
    def __init__(self, img_size=224, patch_size=16, num_classes=10, embed_dim=768, num_heads=8, depth=6, mlp_dim=1024):
        super().__init__()
        self.patch_embedding = PatchEmbedding(img_size, patch_size, 3, embed_dim)
        self.pos_encoding = PositionalEncoding(embed_dim, (((img_size -patch_size) // patch_size) + 1) ** 2)
        self.transformer_blocks = nn.ModuleList([
            TransformerEncoderBlock(embed_dim, num_heads, mlp_dim) for _ in range(depth)
        ])
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.mlp_head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        B = x.size(0)
        x = self.patch_embedding(x)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = self.pos_encoding(x)
        for block in self.transformer_blocks:
            x = block(x)
        return self.mlp_head(x[:, 0, :])

In [ ]:
class VisionTransformer(nn.Module):
    def __init__(self, img_size=224, patch_size=16, num_classes=10, embed_dim=768, num_heads=8, depth=6, mlp_dim=1024):
        super().__init__()


    def forward(self, x):

        return self.mlp_head(x[:, 0, :])

In [6]:
!pip install sympy==1.13.3 --break-system-packages

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 63.1 MB/s eta 0:00:00
  Attempting uninstall: sympy
    Found existing installation: sympy 1.14.0
    Uninstalling sympy-1.14.0:
      Successfully uninstalled sympy-1.14.0


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [8]:
from tqdm import tqdm
import torch.optim as optim
import sympy
import sympy.printing
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_data = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_data, batch_size=32, shuffle=True)

model = VisionTransformer().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
for epoch in range(5):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/5", leave=True)

    for inputs, labels in loop:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Calculate accuracy for this batch
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        # Show running loss and accuracy on progress bar
        loop.set_postfix(loss=loss.item(), acc=f"{correct/total*100:.2f}%")

    avg_loss = running_loss / len(train_loader)
    train_acc = correct / total * 100
    print(f"Epoch {epoch+1}/5 | Avg Loss: {avg_loss:.4f} | Train Acc: {train_acc:.2f}%")

100%|██████████| 170M/170M [00:03<00:00, 43.0MB/s]
Epoch 1/5: 100%|██████████| 1563/1563 [09:51<00:00,  2.64it/s, acc=29.66%, loss=2.12]


Epoch 1/5 | Avg Loss: 2.2013 | Train Acc: 29.66%


Epoch 2/5: 100%|██████████| 1563/1563 [09:56<00:00,  2.62it/s, acc=29.99%, loss=4.58]


Epoch 2/5 | Avg Loss: 5.4215 | Train Acc: 29.99%


Epoch 3/5: 100%|██████████| 1563/1563 [09:55<00:00,  2.62it/s, acc=25.47%, loss=1.91]


Epoch 3/5 | Avg Loss: 2.1619 | Train Acc: 25.47%


Epoch 4/5: 100%|██████████| 1563/1563 [09:54<00:00,  2.63it/s, acc=27.09%, loss=2.17]


Epoch 4/5 | Avg Loss: 2.5197 | Train Acc: 27.09%


Epoch 5/5: 100%|██████████| 1563/1563 [09:53<00:00,  2.63it/s, acc=25.80%, loss=2.01]

Epoch 5/5 | Avg Loss: 2.2356 | Train Acc: 25.80%
